<a href="https://colab.research.google.com/github/karamdh/arene-des-algos-KARAM-DHIFI/blob/main/jour3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# PHASE A : Régression — California Housing
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

def charger_immobilier():
    data = fetch_california_housing()
    X, y = data.data, data.target
    print(f"California Housing : {X.shape}, cible = prix médian en centaines de milliers de $")
    print(f"Variables : {data.feature_names}")
    return X, y

def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    return {
        "r2"  : r2_score(y_test, y_pred),
        "mae" : mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
    }

X, y = charger_immobilier()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    scores = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<20} R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")


California Housing : (20640, 8), cible = prix médian en centaines de milliers de $
Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
LinearRegression     R2=0.58  MAE=0.53  RMSE=0.75
RandomForest         R2=0.81  MAE=0.33  RMSE=0.51


In [5]:
# CHECKPOINTS PHASE A

print("CHECKPOINT 1 — Dataset complet")
for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    scores = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<20} R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

print()

print("CHECKPOINT 2 — Seulement 100 lignes d'entraînement")
X_100 = X_train_s[:100]
y_100 = y_train[:100]

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest",     RandomForestRegressor(n_estimators=100, random_state=42)),
]:
    scores = evaluer_regression(modele, X_100, X_test_s, y_100, y_test)
    print(f"{nom:<20} R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

print("""
Observation : le R2 s'effondre (peut devenir négatif sur le Random Forest).
Parceque Avec 100 lignes sur 20 640, le modèle n'a pas vu assez de cas
pour généraliser. Le Random Forest surfit : il mémorise les 100 exemples
au lieu d'apprendre une vraie règle. La régression linéaire résiste mieux
car elle a moins de paramètres à estimer.
""")
print()

print("CHECKPOINT 3 — Quartier fictif hors plage")

lr_final = LinearRegression()
lr_final.fit(X_train_s, y_train)


quartier_fictif = np.array([[0, 20, 5, 1, 9000, 3, 37.0, -120.0]])

quartier_fictif_s = scaler.transform(quartier_fictif)

pred_lr = lr_final.predict(quartier_fictif_s)[0]

rf_final = RandomForestRegressor(n_estimators=100, random_state=42)
rf_final.fit(X_train_s, y_train)
pred_rf = rf_final.predict(quartier_fictif_s)[0]

print(f"Quartier fictif : revenu=0, population=9000")
print(f"LinearRegression → prix prédit : {pred_lr:.2f} (x100k$)")
print(f"RandomForest     → prix prédit : {pred_rf:.2f} (x100k$)")

if pred_lr < 0:
    print(f"  Régression linéaire : prix NÉGATIF ({pred_lr:.2f}) — valeur absurde !")
else:
    print(f" Valeur hors plage des données d'entraînement — à surveiller")

print(f"Plage normale des prix dans le dataset : "
      f"{y.min():.2f} à {y.max():.2f} (x100k$)")

print("""
Que faire en production ?
1. Valider les entrées AVANT le modèle : rejeter tout revenu < 0
   ou toute population hors de la plage vue à l'entraînement.
2. Ajouter une couche de détection d'anomalies en amont.
3. Ne jamais faire confiance à une prédiction sur une entrée
   que le modèle n'a jamais vue : extrapoler, c'est risqué.
""")

CHECKPOINT 1 — Dataset complet
LinearRegression     R2=0.58  MAE=0.53  RMSE=0.75
RandomForest         R2=0.81  MAE=0.33  RMSE=0.51

CHECKPOINT 2 — Seulement 100 lignes d'entraînement
LinearRegression     R2=0.40  MAE=0.54  RMSE=0.88
RandomForest         R2=0.54  MAE=0.56  RMSE=0.77

Observation : le R2 s'effondre (peut devenir négatif sur le Random Forest).
Parceque Avec 100 lignes sur 20 640, le modèle n'a pas vu assez de cas
pour généraliser. Le Random Forest surfit : il mémorise les 100 exemples
au lieu d'apprendre une vraie règle. La régression linéaire résiste mieux
car elle a moins de paramètres à estimer.


CHECKPOINT 3 — Quartier fictif hors plage
Quartier fictif : revenu=0, population=9000
LinearRegression → prix prédit : -0.18 (x100k$)
RandomForest     → prix prédit : 0.71 (x100k$)
  Régression linéaire : prix NÉGATIF (-0.18) — valeur absurde !
Plage normale des prix dans le dataset : 0.15 à 5.00 (x100k$)

Que faire en production ?
1. Valider les entrées AVANT le modèle : rej